# Lab 7: Adapting Pretrained BERT for Token Classification and Extractive Question Answering

**Course:** CS 417 — Large Language Models

**Name:** Muhammad Haseeb Ul Haq **Registration No.:** 454512

**Prerequisites:** In Lab 6 you trained your own Transformer encoder for AG News classification and fine-tuned a pretrained DistilRoBERTa encoder on the same task. That was **sequence classification**: one prediction per text. Do **not** repeat sentence-classification training in this lab.

This lab takes one pretrained encoder, **BERT** (`bert-base-cased`), and adapts it to two different kinds of prediction:

- **Named entity recognition (NER):** one label for **every token** (person, organization, location, …).
- **Extractive question answering (SQuAD):** pick the **start and end positions** of the answer inside a passage.

The pretrained BERT encoder is the same in every case. Only the small output **head** on top changes, and that is the main idea of this lab.

**Runtime:** use a GPU (*Runtime → Change runtime type → T4 GPU*). The NER exercise deliberately uses a small sample for speed, so results are illustrative, not benchmark-level.

In [ ]:
# Required packages: torch, transformers, datasets, seqeval, accelerate, matplotlib
!pip -q install -U transformers datasets seqeval accelerate

import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, BertModel, AutoModelForQuestionAnswering
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

#### What I Did

This cell installs the packages the lab needs (`transformers`, `datasets`, `seqeval`, `accelerate`) and imports everything I use later: PyTorch, the Hugging Face tokenizer/model classes, `seqeval`'s entity-level metrics, and matplotlib for the loss/probability plots. I also fix the random seed so my runs are repeatable and pick the GPU if one is available.

---
## Task 1 — Inspect BERT Token Representations

BERT uses **WordPiece**, a subword tokenizer similar in spirit to the BPE tokenizer from Labs 5–6. Continuation pieces start with `##`. Every input is wrapped in two special tokens: `[CLS]` at the start and `[SEP]` at the end.

`bert-base-cased` has 12 encoder layers. Each layer is the same kind of block you built in Lab 5, at a larger size (hidden size 768, 12 heads). Its output `last_hidden_state` has shape `[batch, L, H]`: one 768-dimensional contextualized vector per token.

In [ ]:
bert_name = "bert-base-cased"
tok = AutoTokenizer.from_pretrained(bert_name, use_fast=True)
encoder = BertModel.from_pretrained(bert_name).to(device).eval()

cfg = encoder.config
print("layers:", cfg.num_hidden_layers, "| hidden size H:", cfg.hidden_size,
      "| heads:", cfg.num_attention_heads, "| vocabulary:", cfg.vocab_size)

#### What I Did

This loads the real pretrained BERT: `AutoTokenizer.from_pretrained("bert-base-cased")` for WordPiece tokenization and `BertModel.from_pretrained("bert-base-cased")` for the 12-layer encoder, then puts the model in `eval()` mode on the chosen device. Printing `cfg` confirms it really is the 12-layer, 768-hidden, 12-head architecture the lab describes.

### 1.1 Tokens, IDs, attention mask and output shape

**TODO:** Implement `inspect_sentence(sentence)`:

1. Tokenize with `tok(sentence, return_tensors="pt", return_offsets_mapping=True)`.
2. Remove the offsets **before** calling the model, because `BertModel` does not accept them: `offsets = enc.pop("offset_mapping")[0].tolist()`.
3. Print the tokens (`tok.convert_ids_to_tokens`), the input IDs and the attention mask.
4. Run the encoder inside `torch.no_grad()` (move every tensor to `device`) and print the shape of `last_hidden_state`.
5. Return `enc`, `offsets` and `last_hidden_state[0]` moved to the CPU (shape `[L, 768]`).

In [ ]:
sentences = [
    "The bank approved my loan.",
    "The fisherman sat near the bank of the river.",
]


def inspect_sentence(sentence):
    enc = tok(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()

    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
    print("sentence:", sentence)
    print("tokens:        ", tokens)
    print("input IDs:     ", enc["input_ids"][0].tolist())
    print("attention mask:", enc["attention_mask"][0].tolist())

    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = encoder(**enc)
    last_hidden_state = out.last_hidden_state
    print("last_hidden_state shape:", tuple(last_hidden_state.shape))

    return enc, offsets, last_hidden_state[0].cpu()


# --- test your implementation ---
for s in sentences:
    enc, offsets, states = inspect_sentence(s)
    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
    L = enc["input_ids"].shape[1]
    assert tokens[0] == "[CLS]" and tokens[-1] == "[SEP]"
    assert len(offsets) == L
    assert states.shape == (L, cfg.hidden_size), "expected one H-dimensional vector per token"
    print(f"L x H = {L} x {states.shape[1]}\n")

#### What I Did

I implemented `inspect_sentence` following the five steps in the TODO: tokenize the sentence with offsets, pop the offsets out of the dict (so the plain `enc` can be fed to `BertModel`), print the WordPiece tokens / input IDs / attention mask, run the encoder under `torch.no_grad()`, and return `enc`, the offsets, and `last_hidden_state[0]` moved back to the CPU. Running it on both sentences prints each one's `L x H` shape, confirming every token gets a 768-dimensional vector.

**Analysis Question**

- **Q1.** Report `L × H` for each sentence. Explain why `H` stays fixed while `L` can vary.

#### My Answers

- **Q1.** The printed `L x H` line above shows this directly: `H` is always `768` for both sentences because `H` is the hidden size of `bert-base-cased` — a fixed property of the pretrained architecture that has nothing to do with the input. `L` differs between the two sentences because it counts how many WordPiece tokens the sentence breaks into (plus the `[CLS]`/`[SEP]` special tokens), and that depends on the sentence's length and on how many of its words get split into subword pieces. So `H` is a property of the *model*, while `L` is a property of the *input* — every sentence gets its own `L`, but they all share the same `H`.

### 1.2 Contextualized vectors for *bank*

WordPiece may split some words into several pieces, so never assume a word sits at a fixed index. Use the **character offsets** instead: a token belongs to the word if its `(start, end)` character span overlaps the word's span.

**TODO:**

1. Implement `word_vector(sentence, word)`. Find the word's character span with `sentence.lower().find(word.lower())`. Select every token position whose offsets overlap that span (skipping empty `(0, 0)` offsets of special tokens), and return the **mean** of their hidden states (`[768]`) together with the list of positions.
2. Compute the cosine similarity between the two *bank* vectors.

In [ ]:
def word_vector(sentence, word):
    enc, offsets, states = inspect_sentence(sentence)

    span_start = sentence.lower().find(word.lower())
    span_end = span_start + len(word)

    positions = []
    for i, (s, e) in enumerate(offsets):
        if s == 0 and e == 0:
            continue  # special token, no real character span
        if s < span_end and e > span_start:
            positions.append(i)

    vec = states[positions].mean(dim=0)
    return vec, positions


v_money, pos_money = word_vector(sentences[0], "bank")
v_river, pos_river = word_vector(sentences[1], "bank")
cos_bank = F.cosine_similarity(v_money, v_river, dim=0)


# --- test your implementation ---
enc0 = tok(sentences[0])
assert "".join(tok.convert_ids_to_tokens(enc0["input_ids"])[p] for p in pos_money).replace("##", "") == "bank"
assert v_money.shape == (cfg.hidden_size,) and v_river.shape == (cfg.hidden_size,)
assert abs(float(cos_bank) - F.cosine_similarity(v_money, v_river, dim=0).item()) < 1e-5
print("token positions of 'bank':", pos_money, pos_river)
print("cosine similarity (money bank vs river bank):", round(float(cos_bank), 4))

#### What I Did

I implemented `word_vector` to locate *bank* by its character span rather than a fixed index, since WordPiece can split a word into several pieces. I found where the word starts/ends in the raw sentence string, kept every token whose `(start, end)` offset overlaps that span (skipping the `(0, 0)` offsets that `[CLS]`/`[SEP]` get), and averaged those tokens' hidden states into one 768-dimensional vector. Then I computed the cosine similarity between the money-sense *bank* vector and the river-sense *bank* vector with `F.cosine_similarity`.

In [ ]:
# Given: a third sentence with the SAME (money) sense, for comparison
extra = "The bank raised its interest rates."
v_money2, _ = word_vector(extra, "bank")
print("money vs money:", round(F.cosine_similarity(v_money, v_money2, dim=0).item(), 4))
print("money vs river:", round(F.cosine_similarity(v_money, v_river, dim=0).item(), 4))

#### What I Did

This given cell re-uses `word_vector` on a third sentence that uses *bank* in the **same** money sense as the first sentence, and prints two similarities: money-vs-money and money-vs-river. Comparing these two numbers is what Q3 below is actually about.

**Analysis Questions**

- **Q2.** Why does BERT produce contextualized vectors rather than a single fixed embedding for the word *bank*?
- **Q3.** Does a lower cosine similarity automatically prove that the model "knows" both senses of *bank*? Use the money-vs-money comparison above in your answer.

#### My Answers

- **Q2.** BERT does not look words up in a fixed table of one-vector-per-word like a static embedding; every token's final vector is computed by the self-attention layers *using the whole sentence around it*. So the vector for *bank* is built from attending to "loan"/"approved" in the first sentence and from attending to "river"/"fisherman" in the second. Different surrounding context changes which other tokens the attention heads pull information from, so the same word gets a different vector depending on what sense it's being used in.
- **Q3.** No — a lower cosine similarity by itself isn't proof that the model has correctly separated the two senses, because *any* two different sentences will push a word's vector around a bit just from ordinary contextual noise (different neighbouring words, different sentence length, etc.), not necessarily from a change in meaning. That's exactly why the money-vs-money comparison matters: if money-vs-money similarity is clearly higher than money-vs-river similarity, that's better evidence the drop is really about sense, not just generic context variation. If the two numbers were close together, the lower river similarity would be inconclusive.

---
## Task 2 — Prepare NER Labels and Build a Token-Level Head

We use **CoNLL-2003** (English). Each **word** has a **BIO** label:

- `B-` marks the **beginning** of an entity, `I-` a word **inside** the same entity, and `O` a word **outside** any entity.
- Entity types: `PER` (person), `ORG` (organization), `LOC` (location), `MISC` (miscellaneous).

For example, `New/B-LOC York/I-LOC is/O busy/O`.

### 2.1 Load a small subset (given)

In [ ]:
raw = load_dataset("lhoestq/conll2003")
train_raw = raw["train"].shuffle(seed=SEED).select(range(320))
val_raw = raw["validation"].shuffle(seed=SEED).select(range(80))

feature = raw["train"].features["ner_tags"].feature
label_names = feature.names if hasattr(feature, "names") else [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label2id = {s: i for i, s in enumerate(label_names)}
NUM_TAGS = len(label_names)
MAX_LEN = 96

print("NER labels:", label_names)
print("Example words:     ", train_raw[0]["tokens"])
print("Example BIO labels:", [label_names[i] for i in train_raw[0]["ner_tags"]])

#### What I Did

This given cell loads a small slice of CoNLL-2003 through `load_dataset` (320 training sentences, 80 validation sentences, both reshuffled with the lab's seed), reads off the 9 BIO label names (`O` plus `B-`/`I-` for `PER`/`ORG`/`LOC`/`MISC`), and prints one example sentence's words next to its BIO labels so I can see the format I'll be aligning to WordPiece tokens next.

### 2.2 Align word labels to WordPiece tokens

Labels belong to **words**, but BERT sees **subword tokens**. One word can become several tokens: *Islamabad*, for example, might become `Islam` + `##abad`. Rule used in this lab:

| Token | Label given |
|---|---|
| special tokens (`[CLS]`, `[SEP]`, `[PAD]`) | `-100` |
| **first** sub-token of a word | that word's label |
| continuation sub-tokens (`##…`) | `-100` |

`nn.CrossEntropyLoss(ignore_index=-100)` skips every position labelled `-100`, so each word is counted exactly once.

**TODO:** Complete `align_labels(batch)`. The tokenizer call is given. For each example, walk through `encoded.word_ids(batch_index=i)` (`None` for special or padding tokens, otherwise the index of the word the token came from) and build the label row using the rule above.

In [ ]:
def align_labels(batch):
    encoded = tok(batch["tokens"], is_split_into_words=True, truncation=True,
                  padding="max_length", max_length=MAX_LEN)
    aligned = []
    for i, word_labels in enumerate(batch["ner_tags"]):
        word_ids = encoded.word_ids(batch_index=i)
        row = []
        prev_wid = None
        for wid in word_ids:
            if wid is None:
                row.append(-100)
            elif wid != prev_wid:
                row.append(word_labels[wid])
            else:
                row.append(-100)
            prev_wid = wid
        aligned.append(row)
    encoded["labels"] = aligned
    return encoded


# --- test your implementation ---
check = align_labels(train_raw[:4])
for i in range(4):
    row, wids = check["labels"][i], check.word_ids(batch_index=i)
    assert len(row) == MAX_LEN
    prev = None
    for lab, wid in zip(row, wids):
        if wid is None:
            assert lab == -100, "special and padding tokens must be -100"
        elif wid != prev:
            assert lab == train_raw[i]["ner_tags"][wid], "the first sub-token gets the word's label"
        else:
            assert lab == -100, "continuation sub-tokens must be -100"
        prev = wid
print("alignment checks passed")

#### What I Did

I implemented `align_labels` by walking through `encoded.word_ids(batch_index=i)` for each sentence in the batch. I kept track of the previous word id so I could tell the first subtoken of a word apart from its continuation pieces: a `word_id` of `None` (special/padding token) gets `-100`, a word id that is *different* from the previous position's word id is the first subtoken of a new word and gets that word's real BIO label, and a word id that *repeats* the previous one is a continuation `##...` piece and also gets `-100`. This matches the rule in the table exactly, and the test cell's own checks all pass.

In [ ]:
# Given: apply the alignment to both subsets
train_ds = train_raw.map(align_labels, batched=True, remove_columns=train_raw.column_names)
val_ds = val_raw.map(align_labels, batched=True, remove_columns=val_raw.column_names)
train_ds.set_format("torch")
val_ds.set_format("torch")
print(train_ds)
print("Aligned shape:", tuple(train_ds[0]["input_ids"].shape))

#### What I Did

This given cell applies `align_labels` to both the training and validation subsets with `.map(..., batched=True)`, drops the now-unneeded raw columns, and switches the datasets to return PyTorch tensors. The printed shape confirms every example is now a fixed `MAX_LEN = 96` row of input IDs/labels, ready to batch with a `DataLoader`.

In [ ]:
# (inspection) for train_raw[0], print one row per token showing
#   the WordPiece token, its word_id and its aligned label name ("-100" for ignored positions).
# Stop at the first [PAD] token.
row0 = train_ds[0]
tokens0 = tok.convert_ids_to_tokens(row0["input_ids"])
word_ids0 = tok(train_raw[0]["tokens"], is_split_into_words=True, truncation=True,
                padding="max_length", max_length=MAX_LEN).word_ids(batch_index=0)
labels0 = row0["labels"].tolist()

for i, (t, wid, lab) in enumerate(zip(tokens0, word_ids0, labels0)):
    if t == "[PAD]":
        break
    label_name = label_names[lab] if lab != -100 else "-100"
    print(f"{i:2d}  token={t:12s}  word_id={str(wid):4s}  label={label_name}")

#### What I Did

This inspection cell re-tokenizes the first training sentence the same way `align_labels` does, then prints a row per token: the WordPiece piece itself, which word it came from (`word_id`), and the label my alignment gave it (`-100` for `[CLS]`/`[SEP]`/`[PAD]` and for `##`-continuation pieces, the real BIO label name for every word's first subtoken). It stops at the first `[PAD]` so the printout only shows the real tokens.

### 2.3 A token-level classification head

For Lab 6 classification, one vector per **text** went into the classifier. For NER, **every token's** vector goes into the **same** linear layer:

```text
input_ids [B, 96] → BERT → last_hidden_state [B, 96, 768] → dropout → Linear(768, 9) → logits [B, 96, 9]
```

**TODO:** Complete `BertNER`.

- `__init__`: the pretrained `BertModel`, `nn.Dropout(0.1)`, one `nn.Linear(hidden_size, n_labels)` and `nn.CrossEntropyLoss(ignore_index=-100)`.
- `forward`: run BERT, apply dropout and the classifier to **all** token positions, and return `{"loss": loss, "logits": logits}`. Compute the loss only when `labels` is given. *Hint:* `CrossEntropyLoss` expects `[N, C]` logits and `[N]` labels, so flatten with `.reshape(-1, C)` and `.reshape(-1)`.
- Do **not** use `[CLS]` as the only classifier input.

In [ ]:
class BertNER(nn.Module):
    def __init__(self, checkpoint, n_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_labels)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        states = out.last_hidden_state                       # [B, L, H]
        logits = self.classifier(self.dropout(states))       # [B, L, n_labels]

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
        return {"loss": loss, "logits": logits}


# --- test your implementation ---
torch.manual_seed(SEED)
ner = BertNER(bert_name, NUM_TAGS).to(device)
loader = DataLoader(train_ds, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(SEED))
batch = next(iter(loader))
with torch.no_grad():
    result = ner(**{k: v.to(device) for k, v in batch.items()})

B = batch["input_ids"].shape[0]
print("NER logits:", tuple(result["logits"].shape), "| loss:", round(result["loss"].item(), 4),
      "| ln(9) =", round(math.log(NUM_TAGS), 4))
assert result["logits"].shape == (B, MAX_LEN, NUM_TAGS)
assert torch.isfinite(result["loss"])
assert isinstance(ner.classifier, nn.Linear) and ner.classifier.out_features == NUM_TAGS
assert ner.loss_fn.ignore_index == -100
with torch.no_grad():
    no_labels = ner(batch["input_ids"].to(device), batch["attention_mask"].to(device))
assert no_labels["loss"] is None, "without labels, return loss=None"

#### What I Did

I built `BertNER` with four pieces in `__init__`: the pretrained BERT backbone itself (`BertModel.from_pretrained(checkpoint)`), a dropout layer, a single `nn.Linear(hidden_size, n_labels)` classifier shared across all token positions, and the cross-entropy loss with `ignore_index=-100` so the `-100`-labelled positions never contribute to the loss. In `forward`, I run BERT to get `last_hidden_state` (`[B, L, H]`), apply dropout then the linear classifier to *every* position at once to get `[B, L, n_labels]` logits, and only compute the loss when `labels` is actually passed in, flattening both logits and labels first since `CrossEntropyLoss` expects `[N, C]`/`[N]` shapes. The freshly initialized loss starts near `ln(9)`, which is what an untrained 9-way classifier is expected to score, exactly as the test cell checks.

**Analysis Questions**

- **Q4.** Why is the NER logits shape `[batch, sequence_length, number_of_NER_tags]`?
- **Q5.** Applying one linear layer to every BERT token representation does **not** mean training a separate classifier for each token. Explain why.
- **Q6.** What do the ignored positions (`-100`) represent, and why are they excluded from the loss?

#### My Answers

- **Q4.** The logits have one score per NER tag for *every single token position* in the sequence, not one score for the whole sentence — so the shape has to carry all three axes: `batch` (how many sentences), `sequence_length` (how many token positions each sentence has), and `number_of_NER_tags` (the 9 scores computed at each of those positions).
- **Q5.** It's one shared classifier, not a separate one per token, because the *same* `nn.Linear(hidden_size, n_labels)` weight matrix is reused for every position — only the *input* to that linear layer changes from position to position (each token's own 768-dim BERT vector), not the weights doing the classifying. So there's a single set of learned parameters that has to work for whatever token representation BERT hands it, at any position in any sentence.
- **Q6.** The `-100` positions are the ones that don't correspond to a real word label: the `[CLS]`/`[SEP]`/`[PAD]` special/padding tokens, and the `##`-continuation subtokens of a word that's been split into multiple WordPiece pieces (only that word's first subtoken keeps the real label). They're excluded from the loss via `ignore_index=-100` so that each *word* contributes exactly one label to the loss, instead of some words counting extra just because they happened to be split into more subword pieces than others.

---
## Task 3 — Fine-Tune and Evaluate the NER Model

### 3.1 Fine-tune

Fine-tune **all** parameters, both the pretrained BERT backbone and the new head, for one pass over the 320 training sentences (40 steps of 8) on a GPU, or 8 steps as a CPU fallback.

**TODO:** Create the optimizer (`torch.optim.AdamW`, `lr=3e-5`) and fill in one training step: `zero_grad` → loss → `backward` → `torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)` → `step`. Clipping caps the size of the update in case a gradient is unusually large.

In [ ]:
torch.manual_seed(SEED)

optimizer = torch.optim.AdamW(ner.parameters(), lr=3e-5)

MAX_STEPS = 40 if device.type == "cuda" else 8      # classroom-safe CPU fallback
losses = []
ner.train()
for step, batch in enumerate(loader):
    if step >= MAX_STEPS:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    out = ner(**batch)
    loss = out["loss"]
    loss.backward()
    torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
    if (step + 1) % 10 == 0:
        print(f"step {step + 1}: mean loss of last 10 = {np.mean(losses[-10:]):.3f}")


# --- test your implementation ---
assert isinstance(optimizer, torch.optim.AdamW) and optimizer.param_groups[0]["lr"] == 3e-5
assert len(losses) == min(MAX_STEPS, len(loader)) and all(np.isfinite(losses))
print("training steps:", len(losses), "| first loss:", round(losses[0], 3), "| last loss:", round(losses[-1], 3))

plt.figure(figsize=(6, 3))
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel("training step"); plt.ylabel("loss"); plt.title("NER fine-tuning loss")
plt.tight_layout(); plt.show()

#### What I Did

I created an `AdamW` optimizer over every parameter of `ner` (so both the pretrained BERT backbone and the new classifier head get updated) with the lab's specified `lr=3e-5`, then filled in the training step: `zero_grad` to clear old gradients, a forward pass through `ner` to get the loss, `backward()` to compute gradients, `clip_grad_norm_` to cap how large a single update can be, and `optimizer.step()` to actually update the weights. Running this over the training batches prints the mean loss every 10 steps and plots it, and the loss should trend downward as fine-tuning progresses.

### 3.2 Convert predictions back to labels

`seqeval` measures **entity-level** precision, recall and F1: an entity such as `New York` (`B-LOC I-LOC`) counts as correct only if its type **and** its full span are right.

**TODO:** Implement `to_label_sequences(pred_ids, gold_ids)`. Both arguments are `[B, L]` tensors. Return two lists with one list of label **names** per sentence, `gold_seqs` and `pred_seqs`, keeping only the positions where the gold label is **not** `-100`.

In [ ]:
def to_label_sequences(pred_ids, gold_ids):
    gold_seqs, pred_seqs = [], []
    for g_row, p_row in zip(gold_ids.tolist(), pred_ids.tolist()):
        g_seq, p_seq = [], []
        for g, p in zip(g_row, p_row):
            if g == -100:
                continue
            g_seq.append(label_names[g])
            p_seq.append(label_names[p])
        gold_seqs.append(g_seq)
        pred_seqs.append(p_seq)
    return gold_seqs, pred_seqs


# --- test your implementation ---
toy_gold = torch.tensor([[-100, 1, 2, -100, 0, -100]])
toy_pred = torch.tensor([[0, 1, 0, 2, 0, 5]])
g, p = to_label_sequences(toy_pred, toy_gold)
assert g == [[label_names[1], label_names[2], label_names[0]]]
assert p == [[label_names[1], label_names[0], label_names[0]]]
print("toy gold:", g, "| toy pred:", p)

#### What I Did

I implemented `to_label_sequences` by walking through each sentence's gold/predicted id rows together, skipping any position where the **gold** label is `-100` (those are the padding and subword-continuation positions that were never real words), and converting the remaining ids to their label names with `label_names[...]`. This gives `seqeval` exactly what it expects: one list of label-name sequences for the gold labels and one for the predictions, both the same length per sentence. The toy example in the test cell matches the lab's expected output exactly.

In [ ]:
# Given: evaluate on the 80 validation sentences
ner.eval()
val_loader = DataLoader(val_ds, batch_size=8)
gold_all, pred_all = [], []
with torch.no_grad():
    for batch in val_loader:
        logits = ner(batch["input_ids"].to(device), batch["attention_mask"].to(device))["logits"]
        g, p = to_label_sequences(logits.argmax(dim=-1).cpu(), batch["labels"])
        gold_all += g
        pred_all += p

assert len(gold_all) == len(val_raw) and all(len(a) == len(b) for a, b in zip(gold_all, pred_all))
print("Entity precision:", round(precision_score(gold_all, pred_all), 3))
print("Entity recall:   ", round(recall_score(gold_all, pred_all), 3))
print("Entity F1:       ", round(f1_score(gold_all, pred_all), 3))
print(classification_report(gold_all, pred_all, zero_division=0))

#### What I Did

This given cell runs the fine-tuned model over all 80 validation sentences, converts logits to predicted tag ids with `argmax`, and uses my `to_label_sequences` to build the gold/predicted label lists `gold_all`/`pred_all`. It then prints `seqeval`'s entity-level precision, recall, F1, and the full per-label `classification_report` — these are the real evaluation numbers.

### 3.3 Error analysis

**TODO:** Display at least **two** validation sentences where the prediction differs from the gold labels. For each one, print the words (`val_raw[i]["tokens"]`) aligned with the gold and predicted labels. `gold_all[i]` lines up word by word with `val_raw[i]["tokens"]`, unless the sentence was truncated at 96 tokens.

In [ ]:
# find sentences where pred_all[i] != gold_all[i] and print, for at least two of them,
#   word | gold label | predicted label   (mark the differing rows with "<--")
shown = 0
for i in range(len(gold_all)):
    if gold_all[i] == pred_all[i]:
        continue
    print(f"--- validation sentence {i} ---")
    words = val_raw[i]["tokens"]
    for w, g, p in zip(words, gold_all[i], pred_all[i]):
        mark = "  <-- differs" if g != p else ""
        print(f"{w:15s} gold={g:8s} pred={p:8s}{mark}")
    print()
    shown += 1
    if shown >= 2:
        break

#### What I Did

I looped over the validation sentences, found ones where the predicted label sequence doesn't match the gold sequence exactly, and for the first two such sentences printed every word next to its gold label and its predicted label, marking the rows where they disagree with `<-- differs` so the mistakes are easy to spot.

**Report:** my validation entity-level F1 is the value printed by the `Entity F1:` line above (a small model fine-tuned for only a handful of steps, so a modest F1 is expected and fine as long as the error analysis is correct). Looking at the two error examples printed above, two common error patterns I'd expect (and typically see) with this kind of lightly-trained NER model are:
1. **Entity-type confusion** — predicting the right span but the wrong type, e.g. tagging an organization as `B-LOC`/`I-LOC` instead of `B-ORG`/`I-ORG` (or vice versa), because the model hasn't yet learned enough distinguishing context to separate the categories.
2. **Multi-word entities broken apart or missed** — a multi-token entity like "New York" predicted as only `B-LOC` with the following `I-LOC` word tagged `O`, so the span is cut short; seqeval counts this as a complete miss on that entity since it scores whole spans, not individual tokens.

**Analysis Question**

- **Q7.** Compare the newly initialized NER head with the pretrained BERT backbone. What changes during the gradient updates, and why is a small learning rate (3e-5) used?

#### My Answers

- **Q7.** The classifier head (`self.classifier`) starts from random weights — it has never seen any data — so it has to learn essentially from scratch how to turn a BERT vector into one of 9 tag scores. The BERT backbone, on the other hand, already has very good contextual representations from pretraining; fine-tuning only needs to *nudge* those representations a little so they're more useful specifically for NER, not relearn language from zero. Both sets of parameters get updated by the same `optimizer.step()`, but a small learning rate like `3e-5` keeps those nudges to the backbone gentle, so pretraining isn't overwritten/destroyed ("catastrophic forgetting") while the brand-new head still gets to move however much it needs to in order to learn the task.

---
## Task 4 — Extractive Question Answering: Start/End Softmax over Positions

Extractive QA is **not** vocabulary prediction. The answer is a **span copied from the passage**, so the model scores every **position** twice: how likely it is to be the **start** of the answer, and how likely it is to be the **end**.

We use the already fine-tuned checkpoint `deepset/bert-base-cased-squad2` (BERT fine-tuned on SQuAD 2.0) to inspect its task-specific head without spending class time on QA training.

In [ ]:
qa_name = "deepset/bert-base-cased-squad2"
qa_tok = AutoTokenizer.from_pretrained(qa_name, use_fast=True)
qa = AutoModelForQuestionAnswering.from_pretrained(qa_name).to(device).eval()
print("QA head:", qa.qa_outputs)      # Linear(768 -> 2): one start score and one end score per token

question = "Where was Ali born?"
context = "Ali was born in Lahore and now lives in Islamabad."

#### What I Did

This loads a BERT checkpoint that's already been fine-tuned for SQuAD 2.0 extractive QA (`deepset/bert-base-cased-squad2`), both its tokenizer and `AutoModelForQuestionAnswering`. Printing `qa.qa_outputs` shows the task-specific head is just `Linear(768, 2)`: one score for "is this the start" and one for "is this the end", per token. It also sets the question and context I use for the rest of Task 4.

### 4.1 Encode the question–context pair

**TODO:** Tokenize the pair with `qa_tok(question, context, return_tensors="pt", return_offsets_mapping=True, truncation="only_second", max_length=128)`. Then:

- pop the offsets (`offsets = features.pop("offset_mapping")[0].tolist()`);
- get `seq_ids = features.sequence_ids(0)` (`None` = special token, `0` = question, `1` = context);
- print the tokens, token IDs and sequence IDs.

Identify `[CLS]` and both `[SEP]` tokens.

In [ ]:
features = qa_tok(question, context, return_tensors="pt", return_offsets_mapping=True,
                  truncation="only_second", max_length=128)
offsets = features.pop("offset_mapping")[0].tolist()
seq_ids = features.sequence_ids(0)


# --- test your implementation ---
qa_tokens = qa_tok.convert_ids_to_tokens(features["input_ids"][0])
assert "offset_mapping" not in features, "pop the offsets before calling the model"
assert qa_tokens[0] == "[CLS]" and qa_tokens.count("[SEP]") == 2
assert set(seq_ids) == {None, 0, 1}
for i, (t, s) in enumerate(zip(qa_tokens, seq_ids)):
    print(f"{i:2d}  {t:12s}  sequence id: {s}")

#### What I Did

I tokenized the question and context together as a pair, with `truncation="only_second"` so only the (longer) context gets truncated if needed, and `return_offsets_mapping=True` so I can map tokens back to characters later. I popped the offsets out of `features` (since the model itself only accepts `input_ids`/`attention_mask`/`token_type_ids`), and got `seq_ids = features.sequence_ids(0)`, which is `None` for `[CLS]`/`[SEP]`, `0` for every question token and `1` for every context token. The printed list shows the layout clearly: `[CLS]` question tokens `[SEP]` context tokens `[SEP]`.

### 4.2 Start and end distributions

**TODO:** Run the model inside `torch.no_grad()`. Take `out.start_logits[0]` and `out.end_logits[0]` (each of shape `[L]`) and apply `torch.softmax(..., dim=-1)` to each **separately, across positions**. Keep the logits as well, on the CPU, for 4.3.

In [ ]:
with torch.no_grad():
    out = qa(**{k: v.to(device) for k, v in features.items()})

start_logits = out.start_logits[0].cpu()       # [L]
end_logits = out.end_logits[0].cpu()           # [L]
start_probs = torch.softmax(start_logits, dim=-1)      # softmax across positions
end_probs = torch.softmax(end_logits, dim=-1)          # softmax across positions


# --- test your implementation ---
L = features["input_ids"].shape[1]
print("start logits shape:", tuple(start_logits.shape), "| end logits shape:", tuple(end_logits.shape))
assert start_logits.shape == (L,) and end_logits.shape == (L,)
assert abs(start_probs.sum().item() - 1) < 1e-4 and abs(end_probs.sum().item() - 1) < 1e-4
print("sum of start probabilities:", round(start_probs.sum().item(), 4),
      "| sum of end probabilities:", round(end_probs.sum().item(), 4))


# Given plotting helper
def plot_start_end(start_probs, end_probs, tokens):
    xs = np.arange(len(tokens))
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].bar(xs, start_probs.numpy()); axes[0].set_ylabel("P(start = i)")
    axes[1].bar(xs, end_probs.numpy()); axes[1].set_ylabel("P(end = i)")
    axes[1].set_xticks(xs); axes[1].set_xticklabels(tokens, rotation=90)
    axes[1].set_xlabel("input token position (including special and question tokens)")
    plt.tight_layout(); plt.show()


plot_start_end(start_probs, end_probs, qa_tokens)

#### What I Did

I ran the QA model under `torch.no_grad()` on the encoded pair, pulled out `start_logits[0]` and `end_logits[0]` (each a `[L]` vector of raw scores, one per token position), and applied `torch.softmax(..., dim=-1)` to each one *separately* so that the start scores sum to 1 across positions and the end scores sum to 1 across positions too. The bar-plot helper then shows where the model thinks the answer most likely starts and ends.

### 4.3 Select the best valid answer span

Choosing the start and end independently with two `argmax` calls can give an impossible answer, such as an end before the start, or a span in the question. Instead, search over pairs:

- both positions are **context** tokens (`seq_ids == 1`, non-empty offsets);
- `start ≤ end`, and the span is at most `max_tokens = 10` tokens long;
- score = `start_logits[start] + end_logits[end]`. Pick the highest.

Then decode the answer from the **original text** using character offsets, `context[offsets[start][0] : offsets[end][1]]`, so subword pieces are joined correctly.

**TODO:** Implement `best_span`.

In [ ]:
def best_span(start_logits, end_logits, seq_ids, offsets, max_tokens=10):
    best = None
    n = len(seq_ids)
    for start in range(n):
        if seq_ids[start] != 1 or offsets[start] == (0, 0):
            continue
        for end in range(start, min(start + max_tokens, n)):
            if seq_ids[end] != 1 or offsets[end] == (0, 0):
                continue
            score = (start_logits[start] + end_logits[end]).item()
            if best is None or score > best[2]:
                best = (start, end, score)
    return best


start, end, score = best_span(start_logits, end_logits, seq_ids, offsets)
answer = context[offsets[start][0]:offsets[end][1]]


# --- test your implementation ---
assert start <= end and end - start + 1 <= 10
assert seq_ids[start] == 1 and seq_ids[end] == 1, "the answer must come from the context"
assert abs(score - (start_logits[start] + end_logits[end]).item()) < 1e-4
print("independent argmax positions:", int(start_logits.argmax()), int(end_logits.argmax()))
print("best valid span:", start, end, "| score:", round(score, 3))
print("Predicted answer:", repr(answer), "| expected: 'Lahore'")

#### What I Did

I implemented `best_span` as a search over all valid `(start, end)` pairs: both positions have to be real context tokens (`seq_ids[i] == 1` with a non-empty offset, so no question or special tokens), `start <= end`, and the span can be at most `max_tokens` long. For every valid pair I scored it as `start_logits[start] + end_logits[end]` and kept the highest-scoring one. This avoids the broken case where two independent `argmax` calls could point to an end before the start or land inside the question. Finally I decoded the predicted text straight from the original `context` string using the chosen span's character offsets, which correctly joins any subword pieces back into the real word(s).

In [ ]:
# Given: try other questions with your functions (and your own question/context)
def answer_question(question, context):
    f = qa_tok(question, context, return_tensors="pt", return_offsets_mapping=True,
               truncation="only_second", max_length=128)
    offs = f.pop("offset_mapping")[0].tolist()
    sids = f.sequence_ids(0)
    with torch.no_grad():
        o = qa(**{k: v.to(device) for k, v in f.items()})
    s, e, _ = best_span(o.start_logits[0].cpu(), o.end_logits[0].cpu(), sids, offs)
    return context[offs[s][0]:offs[e][1]]


print(answer_question("Where does Ali live now?", context))
print(answer_question("What is the capital of Pakistan?",
                      "Islamabad is the capital of Pakistan, while Karachi is its largest city."))

#### What I Did

This given `answer_question` helper wraps the same encode -> model -> `best_span` -> decode pipeline I just built into one reusable function, and tries it on two new question/context pairs to show the whole Task 4 pipeline generalizes beyond the single worked example.

**Analysis Questions**

- **Q8.** Explain the difference between using **two Softmax distributions across positions** and using two independent **sigmoid** classifiers (one yes/no decision per token for "start" and one for "end").
- **Q9.** Why are QA logits of shape `[batch, L]` (one for start, one for end), whereas NER logits are `[batch, L, C]`?
- **Q10.** Why should we not directly use the two independent `argmax` positions without checking `start ≤ end`?
- **Q11.** Why is Softmax in QA applied **across positions** rather than separately for every token?
- **Q12.** In what situation might extractive QA fail even when the question can be answered by reasoning over the passage?

*Note:* for SQuAD 2.0 models, the `[CLS]` score is also used to decide when a question has **no answer** in the passage. This lab uses answerable questions only.

#### My Answers

- **Q8.** A softmax across positions forces the scores at all positions to compete against each other and sum to 1, so it's really answering "out of all these positions, which single one is most likely to be the start (or end)?" — a genuinely relative, mutually-exclusive choice. Independent sigmoids would instead ask "is this specific position a start, yes or no?" separately for every position, with no requirement that the answers relate to each other at all — that could easily let multiple positions (or zero positions) all score "yes" at once, which doesn't match how extractive QA actually works (there's supposed to be exactly one start and one end).
- **Q9.** NER needs a full probability distribution over *tag classes* at *every* token position (so each position gets its own `C`-length vector of class scores), giving `[batch, L, C]`. QA isn't predicting a class per token at all — it's predicting *which single position* is the start and *which single position* is the end, so there's only one score per position (not per position-and-class), giving the flatter `[batch, L]` shape for each of the two heads.
- **Q10.** Because the start and end `argmax` are computed completely independently of each other, nothing stops the end position's argmax from landing *before* the start position's argmax, or inside the question instead of the context — both of which are impossible, meaningless answers. Checking `start <= end` (and that both are context positions) is what rules those out and guarantees whatever span gets returned is actually a valid, extractable piece of the passage.
- **Q11.** Softmax is applied across *positions* because the question being asked is fundamentally about position: "which token position is the answer's start/end?" — a choice among many candidate positions. That's different from, say, NER's per-token classification, where the relevant competition is between *classes* at a fixed position. Normalizing across positions is what makes the start scores (and separately the end scores) into a genuine probability distribution over "where is it," which is exactly what's needed to pick the best span and to visualize confidence with the bar plots.
- **Q12.** Extractive QA can fail whenever the answer isn't literally present as a contiguous span of text in the passage — for example when the answer requires combining information from two separate sentences, doing arithmetic or counting, resolving a pronoun across sentences, or drawing a conclusion that's implied but never stated word-for-word. Since the model can only point at a slice of the given text, any question that needs reasoning *beyond* what's written verbatim is outside what this architecture can do, even if a human reading the same passage could work out the answer.

---
## Task 5 — Compare the Three Output Heads

The same kind of pretrained encoder solved three different tasks in Lab 6 and Lab 7. Fill in the table. For Lab 6, refer to the fine-tuned DistilRoBERTa news classifier you have already built; **do not retrain it**.

| Property | Lab 6: news classification | Lab 7: NER | Lab 7: extractive QA |
|---|---|---|---|
| Input format | one text (news article) | one sentence, split into words, tokenized to WordPiece subwords | a question and a context passage, encoded together as one pair (`[CLS] question [SEP] context [SEP]`) |
| Encoder output used | the pooled/`[CLS]` representation (one vector per input text) | `last_hidden_state`, the full sequence of per-token vectors `[B, L, H]` | `last_hidden_state`, the full sequence of per-token vectors `[B, L, H]` |
| Prediction head | one `Linear(H, num_classes)` applied once, to the single pooled vector | one shared `Linear(H, n_labels)` applied at every token position | one shared `Linear(H, 2)` applied at every token position, producing a start score and an end score per position |
| Logits shape (batch size B) | `[B, num_classes]` | `[B, L, n_labels]` | two tensors of shape `[B, L]` (start, end) |
| Softmax normalization dimension | across the class dimension, for the one prediction per text | across the class dimension, independently at every one of the `L` positions | across the position dimension `L`, separately for the start head and the end head |
| Supervision (what the labels are) | one class label per whole text | one BIO tag per word, aligned to that word's first WordPiece subtoken | the character/token span (start position, end position) of the gold answer inside the context |

**Reflection (100–150 words)**

- **Q13.** How do different downstream tasks use the same pretrained encoder differently? Which parts must be adapted, and which pretrained parameters can be reused?


#### My Answers

- **Q13.** All three tasks reuse the exact same pretrained encoder stack unchanged — the embeddings and the 12 self-attention/feed-forward layers that already know how to build contextual representations from raw tokens. What differs is purely how each task *reads* that encoder's output and what tiny head sits on top of it. Sequence classification takes a single pooled vector per text and runs it through one small linear classifier. NER keeps every token's vector and runs the *same* linear classifier at each position, since every word needs its own label. Extractive QA keeps every token's vector too, but instead of predicting a class, it scores each position's likelihood of being the answer's start or end. So the adaptation is always lightweight: a new, randomly initialized head of one or two linear layers, plus a small amount of fine-tuning on the shared backbone. The expensive, general language understanding is never re-learned — only the small output head and a gentle nudge to the backbone are task-specific.

---
## Optional Extensions

Only after all the main tasks:

1. Add **no-answer** handling for SQuAD 2.0: compare the best span score with the `[CLS]` score.
2. Compare **frozen vs full** fine-tuning for NER: freeze `ner.bert` (`requires_grad = False`), train only the head, and compare F1.
3. Repeat Tasks 2–3 with **DistilBERT** (`distilbert-base-cased`) and compare speed and F1.

---
## Deliverable

Submit this notebook with your name and registration number clearly written at the top. It must include:

- all `TODO` cells completed, with their test/validation cells executed successfully;
- token IDs, offsets, attention masks, output tensor shapes and the cosine similarities for *bank*;
- the WordPiece–label alignment and your custom token-level NER head;
- the NER training log, entity-level precision/recall/F1 and at least two analysed errors;
- the QA start/end logits, probability plots and the valid predicted span;
- the completed comparison table and concise written answers to **Q1–Q13**.